# 추석 연휴 전국 기상 예측 보도 실습 (Google Colab · 3-셀 구성)

**목표**: 기상청 API 허브의 **중기예보**(4~10일 후 예보)로 2026년 추석 연휴(9/24~9/27)
전국 주요 지역의 기온·강수확률을 수집 → CSV/차트 저장 → 기사 작성용 핵심 수치 요약까지 진행합니다.

## 이 노트북은 셀 3개로 구성되어 있습니다
1. **준비** — 한글 폰트 설치, 라이브러리 임포트
2. **설정 + 데이터 수집** — 인증키/날짜/지역코드 입력부터 전국 데이터 수집까지 한 번에
3. **결과 확인 + 시각화** — CSV 저장, 그래프, 기사용 요약

> ⚠️ **에러가 나면 셀을 하나씩 다시 실행하지 말고, 메뉴에서
> "런타임 → 모두 실행(Run all)"을 눌러 처음부터 다시 실행하세요.**
> (중간 셀만 다시 실행하면 이전 실행 결과가 남아있어 오류의 원인 파악이 더 어려워집니다.)

⚠️ **왜 검색AI가 아니라 파이썬 API인가?**
정확한 예보 수치는 기상청 1차 자료에서 바로 받아야 합니다. "왜 이런 날씨인지" 같은 배경
설명은 이 데이터를 다 뽑은 뒤 검색AI로 보강하면 됩니다.

⚠️ **중기예보는 하루 2번(06시·18시) 갱신**됩니다. 기사를 실제로 내보내기 직전 다시 실행해서
최신 값으로 갱신하고, 기사에는 "몇 월 며칠 몇 시 발표 기준"이라는 문구를 넣으세요.


## 셀 1. 준비: 한글 폰트 설치 + 라이브러리 임포트

In [ ]:
# 코랩(리눅스) 환경에는 한글 폰트가 기본으로 없어서 나눔고딕을 설치합니다.
!apt-get -qq -y install fonts-nanum > /dev/null

import requests
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from datetime import datetime, timedelta, date

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
fm.fontManager.addfont(font_path)
plt.rc("font", family="NanumGothic")
plt.rcParams["axes.unicode_minus"] = False  # 마이너스(-) 기호 깨짐 방지

print("준비 완료. (그래프 글자가 네모(ㅁ)로 깨지면 '런타임 > 런타임 다시 시작' 후 전체를 다시 실행)")


## 셀 2. 설정 + 전국 데이터 수집

인증키/날짜/지역코드 설정부터, API 호출 함수 정의, 전국 지역 수집까지 이 셀 하나로 끝냅니다.
(함수 정의와 함수를 쓰는 코드를 다른 셀로 나눠두면, 함수만 고쳤을 때 재실행을 깜빡하고
"옛날 버전의 함수가 메모리에 남아있는" 문제가 생기기 쉬워서 하나로 합쳤습니다.)


In [ ]:
# -------------------- 2-1. 설정값 --------------------

# 기상청 API 허브에서 발급받은 인증키
AUTH_KEY = "여기에_발급받은_인증키_입력"

# 기사에서 다룰 날짜 (2026년 추석 연휴: 9/24 목 ~ 9/27 일)
TARGET_DATES = [date(2026, 9, 24), date(2026, 9, 25), date(2026, 9, 26), date(2026, 9, 27)]

# 전국 대표 지역의 중기예보 구역코드
#   land: 중기육상예보(강수확률/날씨) 조회용 (도 단위 구역코드)
#   temp: 중기기온예보(최고/최저기온) 조회용 (대표 도시 지점코드)
#   2026.7.1. 전남/광주 행정구역 통합 등으로 구역코드가 바뀌었을 수 있습니다.
#   아래 코드로 안 되면 노트북 맨 아래 '부록' 셀로 최신 코드를 확인하세요.
REGIONS = {
    "서울·인천·경기": {"land": "11B00000", "temp": "11B10101"},
    "강원영서":       {"land": "11D10000", "temp": "11D10301"},
    "강원영동":       {"land": "11D20000", "temp": "11D20501"},
    "충청북도":       {"land": "11C10000", "temp": "11C10301"},
    "대전·세종·충남": {"land": "11C20000", "temp": "11C20401"},
    "전북":          {"land": "11F10000", "temp": "11F10201"},
    "광주·전남":      {"land": "11F20000", "temp": "11F20501"},
    "대구·경북":      {"land": "11H10000", "temp": "11H10701"},
    "부산·울산·경남": {"land": "11H20000", "temp": "11H20201"},
    "제주도":         {"land": "11G00000", "temp": "11G00201"},
}

# -------------------- 2-2. 발표시각(tmFc) + 날짜별 offset 계산 --------------------

now = datetime.now()
if now.hour >= 18:
    tmfc_dt = now.replace(hour=18, minute=0, second=0, microsecond=0)
elif now.hour >= 6:
    tmfc_dt = now.replace(hour=6, minute=0, second=0, microsecond=0)
else:
    tmfc_dt = (now - timedelta(days=1)).replace(hour=18, minute=0, second=0, microsecond=0)

TM_FC = tmfc_dt.strftime("%Y%m%d%H00")
base_date = tmfc_dt.date()
print(f"기준 발표시각(TM_FC): {TM_FC}")

date_offsets = {}
for d in TARGET_DATES:
    offset = (d - base_date).days
    date_offsets[d] = offset
    if 4 <= offset <= 10:
        print(f"- {d.strftime('%m/%d')} : {offset}일 후 예보 사용 가능")
    else:
        print(f"[주의] {d.strftime('%m/%d')} : {offset}일 후 -> 중기예보 범위(4~10일) 밖입니다.")

# -------------------- 2-3. API 호출 함수 --------------------

BASE_URL = "https://apihub.kma.go.kr/api/typ02/openApi/MidFcstInfoService"

def call_api(operation, reg_id, debug=False):
    url = f"{BASE_URL}/{operation}"
    params = {
        "pageNo": 1, "numOfRows": 10, "dataType": "JSON",
        "regId": reg_id, "tmFc": TM_FC, "authKey": AUTH_KEY,
    }
    res = requests.get(url, params=params)

    if debug:
        print(f"--- {operation}({reg_id}) 원본 응답 (status={res.status_code}) ---")
        print(res.text[:800])
        print("---------------------------------------------")

    try:
        data = res.json()
    except ValueError:
        print(f"[오류] {operation}({reg_id}) JSON이 아닌 응답. call_api(..., debug=True)로 원본 확인.")
        return None

    # 기상청 API 허브 게이트웨이 오류 형식: {"result": {"status": 403, "message": "..."}}
    if "result" in data and "response" not in data:
        gw = data["result"]
        print(f"[경고] {operation}({reg_id}) API 허브 오류 {gw.get('status')}: {gw.get('message')}")
        if gw.get("status") == 403:
            print("       -> apihub.kma.go.kr에서 이 API를 별도로 '활용신청'해야 합니다.")
        return None

    header = data.get("response", {}).get("header")
    if header is None:
        alt = data.get("OpenAPI_ServiceResponse", {}).get("cmmMsgHeader", {})
        if alt:
            print(f"[경고] {operation}({reg_id}) 서비스 오류: "
                  f"{alt.get('returnReasonCode')} {alt.get('returnAuthMsg') or alt.get('errMsg')}")
        else:
            print(f"[경고] {operation}({reg_id}) 예상치 못한 응답 형태. call_api(..., debug=True)로 원본 확인.")
        return None

    result_code = header.get("resultCode")
    if result_code != "00":
        print(f"[경고] {operation}({reg_id}) 응답 이상: {result_code} {header.get('resultMsg')}")
        return None
    items = data["response"]["body"]["items"]["item"]
    return items[0] if items else None


def get_day_forecast(land_item, temp_item, offset):
    """offset(며칠 후)에 해당하는 날씨/강수확률/기온을 하나로 정리"""
    result = {
        "최저기온": temp_item.get(f"taMin{offset}") if temp_item else None,
        "최고기온": temp_item.get(f"taMax{offset}") if temp_item else None,
    }
    if land_item is None:
        result["날씨"] = None
        result["강수확률"] = None
        return result

    if 4 <= offset <= 7:
        wf_am, wf_pm = land_item.get(f"wf{offset}Am"), land_item.get(f"wf{offset}Pm")
        rn_am, rn_pm = land_item.get(f"rnSt{offset}Am"), land_item.get(f"rnSt{offset}Pm")
        result["날씨"] = f"오전 {wf_am} / 오후 {wf_pm}" if wf_am != wf_pm else wf_am
        result["강수확률"] = f"오전 {rn_am}% / 오후 {rn_pm}%" if rn_am != rn_pm else f"{rn_am}%"
    else:  # 8~10일 후는 하루 단위
        result["날씨"] = land_item.get(f"wf{offset}")
        result["강수확률"] = f"{land_item.get(f'rnSt{offset}')}%"
    return result

# 아래 한 줄의 주석을 풀면 서울 지역 원본 응답만 먼저 테스트해볼 수 있습니다.
# _ = call_api("getMidLandFcst", REGIONS["서울·인천·경기"]["land"], debug=True)

# -------------------- 2-4. 전국 지역 x 날짜별 수집 --------------------

rows = []
for region_name, codes in REGIONS.items():
    land_item = call_api("getMidLandFcst", codes["land"])
    temp_item = call_api("getMidTa", codes["temp"])

    for d in TARGET_DATES:
        offset = date_offsets[d]
        if not (4 <= offset <= 10):
            continue
        fc = get_day_forecast(land_item, temp_item, offset)
        rows.append({
            "지역": region_name, "날짜": d.strftime("%m/%d"),
            "최저기온": fc["최저기온"], "최고기온": fc["최고기온"],
            "날씨": fc["날씨"], "강수확률": fc["강수확률"],
        })

df = pd.DataFrame(rows)
df["최저기온"] = pd.to_numeric(df["최저기온"], errors="coerce")
df["최고기온"] = pd.to_numeric(df["최고기온"], errors="coerce")
df


## 셀 3. 결과 확인: CSV 저장 + 시각화 + 기사용 요약

In [ ]:
# -------------------- 3-1. CSV 저장 --------------------
csv_filename = f"chuseok_weather_forecast_{TM_FC}.csv"
df.to_csv(csv_filename, index=False, encoding="utf-8-sig")
print(f"저장 완료: {csv_filename}  (왼쪽 폴더 아이콘 -> 파일 목록에서 우클릭 -> 다운로드)")

# from google.colab import files
# files.download(csv_filename)  # 바로 내 컴퓨터로 받고 싶으면 주석 해제

# -------------------- 3-2. 시각화 --------------------
dates_sorted = [d.strftime("%m/%d") for d in TARGET_DATES if 4 <= date_offsets[d] <= 10]
n = len(dates_sorted)
ncols = 2
nrows = (n + 1) // 2

fig, axes = plt.subplots(nrows, ncols, figsize=(13, 5 * nrows), squeeze=False)

for i, day in enumerate(dates_sorted):
    ax = axes[i // ncols][i % ncols]
    sub = df[df["날짜"] == day].dropna(subset=["최저기온", "최고기온"])
    x = range(len(sub))

    ax.bar(x, sub["최고기온"] - sub["최저기온"], bottom=sub["최저기온"],
           color="#e67e22", alpha=0.8, width=0.6)

    for xi, (_, row) in zip(x, sub.iterrows()):
        ax.text(xi, row["최고기온"] + 0.3, f"{row['최고기온']:.0f}°", ha="center", fontsize=9, color="#c0392b")
        ax.text(xi, row["최저기온"] - 0.8, f"{row['최저기온']:.0f}°", ha="center", fontsize=9, color="#2980b9")

    ax.set_xticks(list(x))
    ax.set_xticklabels(sub["지역"], rotation=30, ha="right", fontsize=9)
    ax.set_title(f"{day} 전국 예상 기온", fontsize=13)
    ax.set_ylabel("기온 (℃)")
    ax.grid(axis="y", alpha=0.3)

for j in range(n, nrows * ncols):
    axes[j // ncols][j % ncols].axis("off")

plt.suptitle(f"추석 연휴 전국 기온 전망 (발표: {tmfc_dt.strftime('%m월 %d일 %H시')} 중기예보 기준)", fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig("chuseok_weather_chart.png", dpi=150, bbox_inches="tight")
plt.show()
print("차트 저장 완료: chuseok_weather_chart.png")

# -------------------- 3-3. 기사용 핵심 수치 요약 --------------------
valid = df.dropna(subset=["최저기온", "최고기온"])
print(f"\n=== {tmfc_dt.strftime('%m월 %d일 %H시')} 발표 중기예보 기준 추석 연휴 날씨 요약 ===\n")

for day in dates_sorted:
    sub = valid[valid["날짜"] == day]
    if sub.empty:
        continue
    hottest = sub.loc[sub["최고기온"].idxmax()]
    coldest = sub.loc[sub["최저기온"].idxmin()]
    rainy = sub[sub["강수확률"].str.contains(r"[3-9][0-9]%|100%", na=False, regex=True)]

    print(f"[{day}]")
    print(f"  - 가장 더운 지역: {hottest['지역']} (최고 {hottest['최고기온']:.0f}℃)")
    print(f"  - 가장 추운 지역: {coldest['지역']} (최저 {coldest['최저기온']:.0f}℃)")
    print(f"  - 비 소식 있는 지역: {', '.join(rainy['지역'].tolist()) if not rainy.empty else '없음'}")
    print()

print("※ 이 요약으로 리드 문장부터 써보세요. 배경 설명은 검색AI로 보강하면 좋습니다.")


## 부록 (선택): 지역코드가 안 맞을 때 - 최신 중기예보구역 코드 조회

REGIONS의 코드로 데이터가 안 나오면, 아래 셀로 기상청이 현재 쓰는 지역코드 원본을 확인하세요.


In [ ]:
url = "https://apihub.kma.go.kr/api/typ01/url/fct_medm_reg.php"
res = requests.get(url, params={"tmfc": 0, "authKey": AUTH_KEY})
print(res.text[:3000])
